## Profitability Analysis

Margin visibility is often the missing link between supply chain operations and commercial strategy. High order volume in a category means nothing if those orders are loss-generating — and in complex distribution networks with multiple shipping modes and customer segments, the true margin picture is rarely obvious from topline revenue. This notebook breaks down order-level profitability by segment, region, and category to identify where the business is actually making money and where shipping costs or discounts are eroding margin.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data
from src.feature_engineering import compute_order_profitability_tier
from src.viz_utils import plot_profit_by_segment_region

pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = load_data('../data/dataco_supply_chain.csv')
df = compute_order_profitability_tier(df)

In [ ]:
tier_dist = df['profit_tier'].value_counts(normalize=True).mul(100).round(1)
print("Profit tier distribution:")
print(tier_dist.to_string())
print(f"\nMean profit per order: ${df['order_profit'].mean():.2f}")
print(f"Median profit per order: ${df['order_profit'].median():.2f}")
print(f"Loss-generating orders: {(df['order_profit'] < 0).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'Loss': '#E8735A', 'Breakeven': '#F5B942', 'Low Margin': '#6BAB8A', 'Healthy': '#4A6FA5'}
tier_counts = df['profit_tier'].value_counts()
axes[0].pie(
    tier_counts.values,
    labels=tier_counts.index,
    colors=[colors.get(t, '#ccc') for t in tier_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Order Count by Profit Tier')

df.groupby('profit_tier')['order_profit'].mean().reindex(['Loss','Breakeven','Low Margin','Healthy']).plot.bar(
    ax=axes[1],
    color=[colors.get(t, '#ccc') for t in ['Loss','Breakeven','Low Margin','Healthy']],
    edgecolor='white'
)
axes[1].set_title('Average Profit per Tier')
axes[1].set_ylabel('Avg Profit ($)')
axes[1].set_xlabel('')
axes[1].axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_profit_by_segment_region(df)
fig.show()

The segment × market heatmap reveals which customer segments are profitable in which geographies — and which combinations are margin-negative. This is directly actionable for pricing strategy: markets where Corporate customers generate lower margins than Consumer customers may indicate miscalibrated volume discount schedules.

In [ ]:
cat_profit = (
    df.groupby('category_name')['order_profit']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'avg_profit', 'sum': 'total_profit', 'count': 'orders'})
    .sort_values('avg_profit')
)

top10 = cat_profit.tail(10)
bot10 = cat_profit.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top10['avg_profit'].plot.barh(ax=axes[0], color='#4A6FA5')
axes[0].set_title('Top 10 Categories — Avg Profit/Order')
axes[0].set_xlabel('Avg Profit ($)')

bot10['avg_profit'].plot.barh(ax=axes[1], color='#E8735A')
axes[1].set_title('Bottom 10 Categories — Avg Profit/Order')
axes[1].set_xlabel('Avg Profit ($)')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
segment_mode_profit = (
    df.groupby(['customer_segment', 'shipping_mode'])['order_profit']
    .mean()
    .round(2)
    .unstack('shipping_mode')
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(segment_mode_profit, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Avg Profit per Order: Customer Segment × Shipping Mode')
plt.tight_layout()
plt.show()

> **Insight:** Categories with negative average profit aren't necessarily ones to exit — they may anchor high-margin cross-sell purchases or serve strategic customer retention roles. The more actionable question is whether those categories are also using expensive shipping modes. If a loss-making category is predominantly shipped via Same Day, the shipping cost structure alone may explain the margin erosion, and a mode-shift policy could recover profitability without touching pricing.